In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
# 1. Postavke povezivanja
USER = 'root'
PASSWORD = '3k0p13!4'
HOST = 'localhost'
DB_NAME = 'fipu_srp_projekt'

# Kreiranje engine-a
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB_NAME}")

In [4]:
# 2. Provjera duplikata i ključeva
print("--- 1. PROVJERA INTEGRITETA ---")
with engine.connect() as conn:
    # Provjera duplih ticket_id-ova
    dup_query = text("SELECT id, COUNT(*) FROM support_tickets GROUP BY id HAVING COUNT(*) > 1")
    dups = conn.execute(dup_query).fetchall()
    if not dups:
        print("Nema duplih ticket_id-ova.")
    else:
        print(f"Pronađeni duplikati: {dups}")

--- 1. PROVJERA INTEGRITETA ---
Nema duplih ticket_id-ova.


In [6]:
# 3. Provjera NULL vrijednosti u ključnim stupcima
print("\n--- 2. PROVJERA NULL VRIJEDNOSTI ---")
query = "SELECT * FROM support_tickets"
df = pd.read_sql(query, engine)
    
null_counts = df[['id', 'issue_proj', 'issue_reporter', 'issue_assignee']].isnull().sum()
print("Broj NULL vrijednosti po ključnim stupcima:")
print(null_counts)


--- 2. PROVJERA NULL VRIJEDNOSTI ---
Broj NULL vrijednosti po ključnim stupcima:
id                    0
issue_proj            0
issue_reporter        0
issue_assignee    24771
dtype: int64


In [8]:
# 4. Provjera konzistentnosti deanonimizacije
print("\n--- 3. KONZISTENTNOST DEANONIMIZACIJE ---")
# Provjeravamo ima li ista šifra projekta više različitih imena (što ne bi smjelo biti)
check_query = """
SELECT issue_proj, COUNT(DISTINCT issue_proj) as name_count 
FROM support_tickets 
GROUP BY issue_proj 
HAVING name_count > 1
"""
with engine.connect() as conn:
    inconsistent = conn.execute(text(check_query)).fetchall()
    if not inconsistent:
        print("Mapiranje projekata je konzistentno (1 ID = 1 Ime).")
    else:
        print(f"Pronađena nekonzistentnost u projektima: {inconsistent}")


--- 3. KONZISTENTNOST DEANONIMIZACIJE ---
Mapiranje projekata je konzistentno (1 ID = 1 Ime).


In [11]:
# 5. Distribucija podataka (Data Profiling)
print("\n--- 4. PROFILIRANJE PODATAKA ---")
df = pd.read_sql("SELECT issue_type, issue_priority, issue_status FROM support_tickets", engine)
print("Distribucija prioriteta:")
print(df['issue_priority'].value_counts())


--- 4. PROFILIRANJE PODATAKA ---
Distribucija prioriteta:
issue_priority
unknown    27156
Medium     19901
High        3595
Highest     1636
Blocker      525
Low          469
Lowest        71
Name: count, dtype: int64
